In [1]:
! pip install -q datasets pandas tqdm

In [20]:
from datasets import load_dataset
from openai import OpenAI
import os
import json
import mlflow
import pandas as pd
import time
from utils import generate_urls, calculate_invoice_accuracies, key_level_metrics

from dotenv import load_dotenv
from config import MODEL_URI, MODEL_NAME
from mlflow.metrics.base import MetricValue
from mlflow.models import make_metric
import numpy as np


load_dotenv()

True

### Load the dataset

In [3]:
dataset = load_dataset("naver-clova-ix/cord-v2")
dataset

README.md:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-b4aaeceff1d90e(…):   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00001-of-00004-7dbbe248962764(…):   0%|          | 0.00/441M [00:00<?, ?B/s]

data/train-00002-of-00004-688fe1305a55e5(…):   0%|          | 0.00/444M [00:00<?, ?B/s]

...

data/test-00000-of-00001-9c204eb3f4e1179(…):   0%|          | 0.00/234M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
})

### Initialize MLflow and OpenAI environment

In [4]:
client = OpenAI()
os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:8080/"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "cord-v2-gpt5-medium"
mlflow.openai.autolog()

### Preprocess data

In [5]:
with open("schema.json", "r") as f:
    schema_dict = json.load(f)

schema_dict

{'menu': {'nm': 'name of the menu',
  'num': 'identification number of menu',
  'unitprice': 'unit price of menu',
  'cnt': 'quantity of menu',
  'discountprice': 'discounted price of menu',
  'price': 'total price of menu',
  'itemsubtotal': 'price of each menu after discount applied',
  'vatyn': 'whether the price includes tax or not',
  'etc': 'others',
  'sub': {'nm': 'name of submenu',
   'unitprice': 'unit price of submenu',
   'cnt': 'quantity of submenu',
   'price': 'total price of submenu',
   'etc': 'others'}},
 'sub_total': {'price': 'subtotal price',
  'discount_price': 'discounted price in total',
  'service_price': 'service charge',
  'othersvc_price': 'added charge other than service charge',
  'tax_price': 'tax amount',
  'etc': 'others'},
 'total': {'total_price': 'total price',
  'etc': 'others',
  'cashprice': 'amount of price paid in cash',
  'changeprice': 'amount of change in cash',
  'creditcardprice': 'amount of price paid in credit/debit card',
  'emoneyprice'

In [6]:
NUM_TEST_SAMPLES = 10
test_dataset = dataset["test"].select(range(NUM_TEST_SAMPLES))

url_list, ground_truth_list = generate_urls(dataset=test_dataset)

schema_dict_list = [schema_dict] * len(url_list)

eval_df = pd.DataFrame(
    {
        "schema": schema_dict_list,
        "image_base64": url_list,
    }
)

eval_df


100%|██████████| 10/10 [00:00<00:00, 24.17it/s]


,schema,image_base64
0,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
1,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
2,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
3,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
4,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
5,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
6,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
7,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
8,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
9,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...


### Run Inference

In [7]:
model = mlflow.pyfunc.load_model(MODEL_URI)

2025/08/16 06:34:33 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-1b0300a8d4684d87aadfd2ededb0c29a
2025/08/16 06:34:33 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.


In [8]:
start_time = time.time()
response_list = model.predict(eval_df, params={"temperature": 0})

end_time = time.time()
inference_time = end_time - start_time
print(response_list)

2025/08/16 06:34:40 WARNING mlflow.models.utils: `params` can only be specified at inference time if the model signature defines a params schema. This model does not define a params schema. Ignoring provided params: ['temperature']


['{\n  "menu": {\n    "nm": "-TICKET CP",\n    "num": "901016",\n    "unitprice": "60.000",\n    "cnt": "2",\n    "price": "60.000"\n  },\n  "sub_total": {\n    "price": "60.000",\n    "discount_price": "-60.000",\n    "tax_price": "5.455"\n  },\n  "total": {\n    "total_price": "60.000",\n    "creditcardprice": "60.000",\n    "menuqty_cnt": "2.00"\n  }\n}', '{\n  "menu": [\n    {\n      "nm": "J.STB PROMO",\n      "price": "17500"\n    },\n    {\n      "nm": "Y.B.BAT",\n      "price": "46000"\n    },\n    {\n      "nm": "Y.BASO PROM",\n      "price": "27500"\n    }\n  ],\n  "total": {\n    "total_price": "91000",\n    "cashprice": "91000"\n  }\n}', '{\n  "menu": {\n    "nm": "JASMINE MT (L)",\n    "num": "1",\n    "unitprice": "24.000",\n    "cnt": "1",\n    "price": "24.000",\n    "sub": {\n      "nm": "COCONUT JELLY (L)",\n      "unitprice": "4.000",\n      "price": "4.000"\n    }\n  },\n  "sub_total": {\n    "price": "28.000"\n  },\n  "total": {\n    "total_price": "28.000",\n    "

[Trace(trace_id=tr-b5fd77524857554b1154de7bbece3904), Trace(trace_id=tr-cdc37460e188f08e64506e3cd8ac217f), Trace(trace_id=tr-62ebe2e0d8d5d4f995bbfa251cc6e3ed), Trace(trace_id=tr-bbaecb29deb59581f6e4bf4091a40fc7), Trace(trace_id=tr-9defc4119e867e5db36a8f1c90e8c544), Trace(trace_id=tr-9f89be74f3de40d2de43abcc58bbf06e), Trace(trace_id=tr-97d24aa6108bcb0e71197ba79820e70f), Trace(trace_id=tr-76cd8534de48cd0891c46497bcafb693), Trace(trace_id=tr-9a92df7203dae0c0c46a299c59129cae), Trace(trace_id=tr-17916f8748f97bcde1f767a4f7eab113)]

In [9]:
len(response_list)

10

### Calculate Number of tokens

In [ ]:
# input_tokens = 0
# output_tokens = 0
# reasoning_tokens = 0
# for response in response_list:
#     input_tokens += response.usage.input_tokens
#     output_tokens += response.usage.output_tokens
#     reasoning_tokens += response.usage.output_tokens_details.reasoning_tokens

# print(f"Total input tokens used: {input_tokens}")
# print(f"Total output tokens used: {output_tokens}")
# print(f"Total reasoning tokens used: {reasoning_tokens}")

In [10]:
parsed_response_list = [json.loads(response) for response in response_list]

with open("gpt5_predictions.json", "w") as f:
    json.dump(parsed_response_list, f, indent=4)

with open("gpt5_ground_truth.json", "w") as f:
    json.dump(ground_truth_list, f, indent=4)

In [11]:
ground_truth_list[1], parsed_response_list[1]

({'menu': [{'nm': 'J.STB PROMO', 'price': '17500'},
   {'nm': 'Y.B.BAT', 'price': '46000'},
   {'nm': 'Y.BASO PROM', 'price': '27500'}],
  'total': {'total_price': '91000', 'cashprice': '91000'}},
 {'menu': [{'nm': 'J.STB PROMO', 'price': '17500'},
   {'nm': 'Y.B.BAT', 'price': '46000'},
   {'nm': 'Y.BASO PROM', 'price': '27500'}],
  'total': {'total_price': '91000', 'cashprice': '91000'}})

### Evaluation

In [12]:
invoice_metrics_df = calculate_invoice_accuracies(ground_truth_list, parsed_response_list)
print(invoice_metrics_df)
print("Average accuracy:", invoice_metrics_df["accuracy"].mean())

   invoice_no  total_keys  matched_keys  accuracy
0           0          11             9  0.818182
1           1           8             8  1.000000
2           2          10             6  0.600000
3           3           8             6  0.750000
4           4          14            13  0.928571
5           5           7             5  0.714286
6           6          12            11  0.916667
7           7           8             1  0.125000
8           8          14             3  0.214286
9           9           6             5  0.833333
Average accuracy: 0.6900324675324676


In [13]:
key_metrics = key_level_metrics(ground_truth_list, parsed_response_list)
key_metrics_df = pd.DataFrame(key_metrics).T.reset_index(names='key')
key_metrics_df

False negative for key 'menu.itemsubtotal': GT='60.000', Pred=None
False positive for key 'menu.unitprice': GT=None, Pred='60.000'
False negative for key 'sub_total.subtotal_price': GT='60.000', Pred=None
False positive for key 'sub_total.price': GT=None, Pred='60.000'
False negative for key 'total.menuqty_cnt': GT='1', Pred=None
False positive for key 'menu.nm': GT='JASMINE MT ( L )', Pred='JASMINE MT (L)'
False positive for key 'total.menutype_cnt': GT=None, Pred='1'
False positive for key 'menu.sub.nm': GT='COCONUT JELLY ( L )', Pred='COCONUT JELLY (L)'
False positive for key 'menu.unitprice': GT=None, Pred='24.000'
False negative for key 'sub_total.subtotal_price': GT='28.000', Pred=None
False positive for key 'menu.num': GT=None, Pred='1'
False positive for key 'menu.sub.unitprice': GT=None, Pred='4.000'
False positive for key 'sub_total.price': GT=None, Pred='28.000'
False positive for key 'total.menuqty_cnt': GT='1.00xITEMs', Pred='1X'
False positive for key 'menu.etc': GT=None,

,key,precision,recall,f1,tp,fp,fn
0,total.creditcardprice,1.000000,1.000000,1.000000,2.0,0.0,0.0
1,total.menuqty_cnt,0.500000,0.333333,0.400000,1.0,1.0,2.0
2,menu.itemsubtotal,0.000000,0.000000,0.000000,0.0,2.0,1.0
3,menu.nm,0.800000,0.666667,0.727273,4.0,1.0,2.0
4,total.total_price,1.000000,1.000000,1.000000,9.0,0.0,0.0
5,sub_total.discount_price,1.000000,1.000000,1.000000,2.0,0.0,0.0
6,menu.unitprice,0.400000,1.000000,0.571429,2.0,3.0,0.0
7,sub_total.subtotal_price,0.000000,0.000000,0.000000,0.0,0.0,8.0
8,menu.cnt,0.750000,0.750000,0.750000,3.0,1.0,1.0
9,menu.num,0.500000,1.000000,0.666667,1.0,1.0,0.0


In [14]:

if not os.path.exists("artifacts"):
    os.makedirs("artifacts")

with mlflow.start_run(run_name=f"{MODEL_NAME}-summary") as run:
    mlflow.log_params({
        "model_name": MODEL_NAME,
    })

    # Log invoice_metrics_df
    invoice_metrics_df.to_csv("artifacts/invoice_metrics.csv", index=False)
    mlflow.log_artifact("artifacts/invoice_metrics.csv")

    key_metrics_df.to_csv("artifacts/key_metrics.csv", index=False)
    mlflow.log_artifact("artifacts/key_metrics.csv")

    # Log the inference time
    mlflow.log_metric("inference_time", inference_time)

    # # log the token usage
    # mlflow.log_metric("input_tokens", input_tokens)
    # mlflow.log_metric("output_tokens", output_tokens)
    

🏃 View run gpt-5-mini-summary at: http://localhost:8080/#/experiments/481396941930403223/runs/6931273895f24bfbb77cad1547f2e15e
🧪 View experiment at: http://localhost:8080/#/experiments/481396941930403223


In [18]:
eval_with_gt_df = pd.concat([pd.DataFrame({"ground_truth": ground_truth_list}), eval_df], axis=1)
eval_with_gt_df

,ground_truth,schema,image_base64
0,"{'menu': {'nm': '-TICKET CP', 'num': '901016',...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
1,"{'menu': [{'nm': 'J.STB PROMO', 'price': '1750...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
2,"{'menu': {'nm': 'JASMINE MT ( L )', 'cnt': '1'...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
3,"{'menu': {'nm': 'DONAT GULA', 'unitprice': '@1...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
4,"{'menu': [{'nm': 'ICE BLACKCOFFE', 'cnt': '2',...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
5,"{'menu': {'nm': 'TRAD KY TOAST CARTE', 'price'...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
6,"{'menu': [{'nm': 'EGG TART', 'cnt': '1', 'pric...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
7,"{'menu': {'nm': 'Kupon 15', 'price': '100,000'...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
8,"{'menu': {'nm': '[REG] BLACK SAKURA', 'cnt': '...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
9,"{'menu': {'nm': 'Bumbu Kaldu Ayam 1', 'unitpri...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...


In [21]:
def business_value_metric(predictions, targets, metrics):
    predictions_list = [json.loads(pred) for pred in predictions]
    targets_list = targets.tolist()
    invoice_metrics_df = calculate_invoice_accuracies(targets_list, predictions_list)
    print(invoice_metrics_df)
    business_value = invoice_metrics_df["accuracy"].mean()
    print("Average accuracy:", business_value)

    return business_value

In [22]:
business_metric = make_metric(
    eval_fn=business_value_metric, greater_is_better=True, name="business_value"
)

In [23]:
results = mlflow.evaluate(
    MODEL_URI,
    eval_with_gt_df,
    targets="ground_truth",  # specify which column corresponds to the expected output
    extra_metrics=[
        business_metric,
    ],
)

results

2025/08/16 06:40:50 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-1b0300a8d4684d87aadfd2ededb0c29a
2025/08/16 06:40:50 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2025/08/16 06:40:50 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2025/08/16 06:40:50 INFO mlflow.models.evaluation.evaluators.default: Computing model predictions.
2025/08/16 06:41:08 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


   invoice_no  total_keys  matched_keys  accuracy
0           0          11             7  0.636364
Average accuracy: 0.6363636363636364
   invoice_no  total_keys  matched_keys  accuracy
0           0          11             7  0.636364
1           1           8             8  1.000000
2           2          10             7  0.700000
3           3           8             5  0.625000
4           4          14            13  0.928571
5           5           7             5  0.714286
6           6          12            11  0.916667
7           7           8             2  0.250000
8           8          14             3  0.214286
9           9           6             5  0.833333
Average accuracy: 0.6818506493506493
🏃 View run righteous-moth-759 at: http://localhost:8080/#/experiments/481396941930403223/runs/e0501dbc265e4a89b6674704d1d6e3b8
🧪 View experiment at: http://localhost:8080/#/experiments/481396941930403223


[Trace(trace_id=tr-7bf767e008aafe424dc94944bc649609), Trace(trace_id=tr-46fdb6065739293ba0cbc0442ddb6cea), Trace(trace_id=tr-140d59557f15efd2be60d81c43c57371), Trace(trace_id=tr-09ddee54d9e6f9094042aa3a146d1662), Trace(trace_id=tr-7f9ce6446384b6867523018436227e5e), Trace(trace_id=tr-f2f41378f55c96b86b03383baf45eb2e), Trace(trace_id=tr-36fda7532afe980965d92c12cb24ecb5), Trace(trace_id=tr-b342f3a067fcdde4b94c73a1cc07d52c), Trace(trace_id=tr-2deeed608a7c72d64c8106e5b5b807d0), Trace(trace_id=tr-2e501eddccb60d6f94f3385d54c5c65f)]